In [1]:
from pyspark.sql import SparkSession 
import pyspark.sql.functions as F 

In [2]:
spark = (
    SparkSession.builder
    .appName("Desafio01")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 17:20:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/11 17:20:42 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/11 17:20:42 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [3]:
df = spark.read.csv("/workspace/datasets/01_fundamentos/clientes.csv", header = True, inferSchema= True)

In [4]:
df.show()

+----------+------------+-----+--------------+---+------------+-----+-----+
|cliente_id|        nome|idade|        cidade| uf|renda_mensal|ativo|score|
+----------+------------+-----+--------------+---+------------+-----+-----+
|         1|Cliente_0001|   25|Belo Horizonte| MG|     1912.68| true|  550|
|         2|Cliente_0002|   26|        Recife| PE|    13651.78| true|  904|
|         3|Cliente_0003|   20|     São Paulo| SP|     1991.65| true|  538|
|         4|Cliente_0004|   56|Rio de Janeiro| RJ|     1937.84| true|  965|
|         5|Cliente_0005|   52|Belo Horizonte| MG|     8422.08|false|  903|
|         6|Cliente_0006|   69|     Fortaleza| CE|    15843.46| true|  732|
|         7|Cliente_0007|   35|     Fortaleza| CE|     4065.41| true|  404|
|         8|Cliente_0008|   42|      Salvador| BA|     3095.82| true|  918|
|         9|Cliente_0009|   69|     Fortaleza| CE|     2216.93|false|  849|
|        10|Cliente_0010|   42|      Salvador| BA|      2800.2| true|  943|
|        11|

In [5]:
print(df)

DataFrame[cliente_id: int, nome: string, idade: int, cidade: string, uf: string, renda_mensal: double, ativo: boolean, score: int]


In [6]:
df.explain(True)

== Parsed Logical Plan ==
Relation [cliente_id#17,nome#18,idade#19,cidade#20,uf#21,renda_mensal#22,ativo#23,score#24] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, uf: string, renda_mensal: double, ativo: boolean, score: int
Relation [cliente_id#17,nome#18,idade#19,cidade#20,uf#21,renda_mensal#22,ativo#23,score#24] csv

== Optimized Logical Plan ==
Relation [cliente_id#17,nome#18,idade#19,cidade#20,uf#21,renda_mensal#22,ativo#23,score#24] csv

== Physical Plan ==
FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,uf#21,renda_mensal#22,ativo#23,score#24] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/01_fundamentos/clientes.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,nome:string,idade:int,cidade:string,uf:string,renda_mensal:double,ativo:boo...



In [7]:
print(type(df))

<class 'pyspark.sql.dataframe.DataFrame'>


In [8]:
print(type(df.idade))

<class 'pyspark.sql.column.Column'>


In [9]:
df_novo = (
    df.select("cliente_id", "nome", "idade", "cidade", "renda_mensal", "ativo")
    .withColumn("renda_anual", (F.col("renda_mensal")) * 12)
    .withColumn(
        "faixa_etaria",
        F.when((F.col("idade") >= 18) & (F.col("idade") < 60), "Adulto")
        .when(F.col("idade") >= 60, "Idoso")
        .otherwise("Menor de idade")
    )
)



In [10]:
df_filtrado = (
    df_novo.filter(
    (F.col("ativo")) &
    (F.col("idade") >= 30)
)
)

In [11]:
df_filtrado.show()

+----------+------------+-----+--------------+------------+-----+------------------+------------+
|cliente_id|        nome|idade|        cidade|renda_mensal|ativo|       renda_anual|faixa_etaria|
+----------+------------+-----+--------------+------------+-----+------------------+------------+
|         4|Cliente_0004|   56|Rio de Janeiro|     1937.84| true|23254.079999999998|      Adulto|
|         6|Cliente_0006|   69|     Fortaleza|    15843.46| true|         190121.52|       Idoso|
|         7|Cliente_0007|   35|     Fortaleza|     4065.41| true|          48784.92|      Adulto|
|         8|Cliente_0008|   42|      Salvador|     3095.82| true|37149.840000000004|      Adulto|
|        10|Cliente_0010|   42|      Salvador|      2800.2| true|33602.399999999994|      Adulto|
|        11|Cliente_0011|   74|Rio de Janeiro|    15718.16| true|188617.91999999998|       Idoso|
|        12|Cliente_0012|   60|      Salvador|     5260.32| true|          63123.84|       Idoso|
|        13|Cliente_

Pergunta 1

Quais linhas do seu código foram Transformations?

Quais foram Actions?

Resposta: Apenas 1 linha foi action (df.show()) as outras foram todas lazys evaluations o spark montou toda arquitetura mas não executou necessariamente.

Pergunta 2

Em que momento o Spark realmente processou os dados?

Resposta: No momento em que foi aplicada a action df.show(), o spark apos ter feito toda a arquitetura das lazy evaluations apos o df.show() que realmente as ações foram executadas

Pergunta 3

Por que criar

df2 = df.select(...)

é melhor do que fazer

df = df.select(...)

na maioria dos casos?

Resposta: Por que é uma melhor pratica criar uma dataframe em cima de um existente, atribuinda uma variavel diferente a ele, torna-se menos confuso saber quem é o original e quem recebeu alteração

Pergunta 4

Explique, com suas palavras, a diferença entre:

print(df)
df.show()
df.explain()

Resposta: print(df) expoe o schema do dataframe, uma lazy evaluation // df.show() imprime o dataframe para o usuario, uma action // df.explain() basicamente é um manual do que foi executado pelo dataframe

In [12]:
spark.stop()